# 黄金内外盘价差分析工具

用于分析伦敦金价与中国AU9999价格之间的价差

## 计算公式

**公式1：换算AU9999价格**
$$AU9999 = \frac{伦敦金价格 \times 汇率}{金衡盎司}$$

其中：金衡盎司 = 31.1035克

**公式2：内外盘价差**
$$价差 = 换算AU9999 - 实际AU9999$$

- 价差 > 0：国内价格偏低
- 价差 < 0：国内价格偏高

In [ ]:
# 常量定义
ORE_TROY_OUNCE = 31.1035  # 金衡盎司 = 31.1035克

## 输入数据

自动获取实时价格数据（从 akshare）：
- 伦敦金价格：国际现货黄金
- 美元人民币汇率：外汇实时行情
- AU9999：上海黄金交易所现货行情

如果获取失败，会回退到默认数值。如需手动修改，请编辑代码中的默认值部分。

In [ ]:
# ========== 自动获取实时价格 ==========
import akshare as ak

def fetch_realtime_prices():
    """获取实时价格数据"""
    london_price = None
    exchange_rate = None
    actual_au9999 = None
    
    try:
        # 获取上海黄金交易所 AU9999 现货数据（元/克）
        try:
            sge_df = ak.spot_hist_sge(symbol="Au99.99")
            if not sge_df.empty and 'close' in sge_df.columns:
                actual_au9999 = float(sge_df.iloc[-1]['close'])
                print(f"  ✓ AU9999数据获取成功: ¥{actual_au9999:.2f}/克")
            else:
                print(f"  ⚠️ AU9999数据格式异常")
        except Exception as e:
            print(f"  ⚠️ AU9999获取失败: {e}")
        
        # 获取外汇行情（美元/人民币）
        try:
            fx_df = ak.fx_spot_quote()
            # 尝试中文列名
            if '货币对' in fx_df.columns and '买报价' in fx_df.columns:
                usd_cny_row = fx_df[fx_df['货币对'] == 'USD/CNY']
                if not usd_cny_row.empty:
                    exchange_rate = float(usd_cny_row.iloc[0]['买报价'])
                    print(f"  ✓ 汇率数据获取成功: {exchange_rate:.4f}")
                else:
                    print(f"  ⚠️ USD/CNY汇率数据未找到")
            # 尝试英文列名
            elif 'code' in fx_df.columns and 'bid' in fx_df.columns:
                usd_cny_row = fx_df[fx_df['code'] == 'USD/CNY']
                if not usd_cny_row.empty:
                    exchange_rate = float(usd_cny_row.iloc[0]['bid'])
                    print(f"  ✓ 汇率数据获取成功: {exchange_rate:.4f}")
                else:
                    print(f"  ⚠️ USD/CNY汇率数据未找到")
            else:
                print(f"  ⚠️ 外汇数据列名不匹配: {fx_df.columns.tolist()}")
        except Exception as e:
            print(f"  ⚠️ 汇率获取失败: {e}")
        
        # 获取国际现货黄金价格（美元/盎司）
        # 由于 currency_bbs_sina 不存在，使用备用方案：通过国内黄金T+D反推估算
        # 公式：伦敦金 ≈ (国内金价 * 31.1035) / 汇率
        if actual_au9999 and exchange_rate:
            try:
                # 使用已获取的AU9999和汇率反推伦敦金价格
                london_price = actual_au9999 * ORE_TROY_OUNCE / exchange_rate
                print(f"  ✓ 伦敦金价格通过AU9999反推: ${london_price:.2f}/盎司")
                print(f"    (基于 AU9999 ¥{actual_au9999:.2f} / 汇率 {exchange_rate:.4f})")
            except Exception as e:
                print(f"  ⚠️ 伦敦金反推计算失败: {e}")
        else:
            print(f"  ⚠️ 无法反推伦敦金价格（缺少AU9999或汇率数据）")
        
        return london_price, exchange_rate, actual_au9999
        
    except Exception as e:
        print(f"获取实时数据时发生错误: {e}")
        return None, None, None

print("正在获取实时价格数据...")
print("-" * 40)

# 尝试获取实时价格
london_price_auto, exchange_rate_auto, actual_au9999_auto = fetch_realtime_prices()

print("-" * 40)

# 如果获取成功则使用实时价格，否则使用默认值
if any([london_price_auto, exchange_rate_auto, actual_au9999_auto]):
    # 手动输入的默认值（作为后备）
    london_price = london_price_auto if london_price_auto else 5311.46
    exchange_rate = exchange_rate_auto if exchange_rate_auto else 6.9217
    actual_au9999 = actual_au9999_auto if actual_au9999_auto else 1182.00
    print(f"✅ 数据获取完成")
    if not london_price_auto:
        print(f"  ⚠️ 伦敦金价格使用默认值: $5311.46/盎司")
    if not exchange_rate_auto:
        print(f"  ⚠️ 汇率使用默认值: 6.9217")
    if not actual_au9999_auto:
        print(f"  ⚠️ AU9999价格使用默认值: ¥1182.00/克")
else:
    # 手动输入的默认值
    london_price = 5311.46      # 伦敦金价格（美元/盎司）
    exchange_rate = 6.9217      # 美元/人民币汇率
    actual_au9999 = 1182.00      # 中国AU9999价格（元/克）
    print(f"⚠️ 无法获取任何实时数据，全部使用默认值")

print(f"\n当前使用的价格:")
print(f"  伦敦金: ${london_price:.2f}/盎司")
print(f"  汇率: {exchange_rate:.4f}")
print(f"  AU9999: ¥{actual_au9999:.2f}/克")
# ===================================

## 计算结果

In [ ]:
# 计算
converted_au9999 = london_price * exchange_rate / ORE_TROY_OUNCE
price_diff = converted_au9999 - actual_au9999
diff_ratio = price_diff / actual_au9999 * 100

# 显示结果
print("=" * 55)
print("                      分析结果")
print("=" * 55)

print(f"\n【输入数据】")
print(f"  伦敦金价格: {london_price:.2f} 美元/盎司")
print(f"  美元/人民币汇率: {exchange_rate:.4f}")
print(f"  中国AU9999价格: {actual_au9999:.2f} 元/克")

print(f"\n【公式1】换算AU9999（元/克）")
print(f"  {london_price:.2f} × {exchange_rate:.4f} ÷ {ORE_TROY_OUNCE}")
print(f"  = {converted_au9999:.2f} 元/克")

print(f"\n【公式2】内外盘价差")
print(f"  {converted_au9999:.2f} - {actual_au9999:.2f}")
print(f"  = {price_diff:+.2f} 元/克")

if price_diff > 0:
    print(f"  → 国内价格偏低（伦敦金换算后更贵）")
elif price_diff < 0:
    print(f"  → 国内价格偏高（伦敦金换算后更便宜）")
else:
    print(f"  → 内外盘价格一致")

print(f"\n【公式3】内外盘价差比")
print(f"  {price_diff:+.2f} ÷ {actual_au9999:.2f} × 100%")
print(f"  = {diff_ratio:+.2f}%")

print("\n" + "=" * 55)

## 快速计算多个场景

输入多组数据批量计算：

In [ ]:
# 批量计算：每组数据为 (伦敦金价格, 汇率, AU9999价格)
scenarios = [
    (5512, 7.0, 1234.00),    # 场景1
    (5600, 7.1, 1250.00),    # 场景2
    (5400, 6.9, 1220.00),    # 场景3
]

# 计算并生成表格数据
table_data = []
for london, rate, au9999 in scenarios:
    converted = london * rate / ORE_TROY_OUNCE
    diff = converted - au9999
    ratio = diff / au9999 * 100
    table_data.append([london, rate, au9999, converted, diff, ratio])

# 创建DataFrame
import pandas as pd
df = pd.DataFrame(table_data, columns=['伦敦金', '汇率', 'AU9999', '换算价', '价差', '价差比'])

# 格式化数值列
df['伦敦金'] = df['伦敦金'].map('{:.2f}'.format)
df['汇率'] = df['汇率'].map('{:.4f}'.format)
df['AU9999'] = df['AU9999'].map('{:.2f}'.format)
df['换算价'] = df['换算价'].map('{:.2f}'.format)
df['价差'] = df['价差'].map('{:+.2f}'.format)
df['价差比'] = df['价差比'].map('{:+.2f}%'.format)

# 简单线框样式
styled = df.style.set_table_styles([
    {'selector': '', 'props': [('border', '1px solid black')]},
    {'selector': 'th', 'props': [('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black')]},
]).hide(axis='index')

display(styled)